# DiffuSVG — LoRA Inference

Runs the fine-tuned **Qwen2-VL-2B + LoRA** adapter to generate SVGs from text prompts.

**Before running:**
1. `Runtime > Change runtime type > T4 GPU`
2. Upload your `qwen2vl_svg_lora.zip` in Cell 3
3. Run all cells top-to-bottom

In [ ]:
#@title ⚙️ Install dependencies (run once)
!pip install -q transformers==4.45.0 accelerate==0.34.0 peft bitsandbytes
!pip install -q cairosvg pillow
import torch
assert torch.cuda.is_available(), '❌ No GPU! Runtime > Change runtime type > T4 GPU'
print(f'✅ GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
#@title 📦 Upload adapter zip
import zipfile, os
from pathlib import Path

ADAPTER_PATH = './qwen2vl_svg_lora/final_adapter'

if Path(ADAPTER_PATH).exists():
    print(f'✅ Adapter already present: {ADAPTER_PATH}')
else:
    from google.colab import files
    print('Select your qwen2vl_svg_lora.zip file:')
    uploaded = files.upload()
    for fname in uploaded:
        if fname.endswith('.zip'):
            with zipfile.ZipFile(fname) as z:
                z.extractall('.')
            print(f'✅ Extracted {fname}')
            break
    if not Path(ADAPTER_PATH).exists():
        raise FileNotFoundError(f'Expected adapter at {ADAPTER_PATH} after extraction')

In [ ]:
#@title ✏️ Configure prompts

PROMPTS = [
    # training-set prompts (should work best)
    'a house with red roof',
    'a coffee cup',
    'a car',
    'a lighthouse',
    'a bicycle',
    'a rocket',
    # unseen prompts (generalisation test)
    'a star',
    'a tree',
    'a flower',
    'a dog',
]

BASE_MODEL  = 'Qwen/Qwen2-VL-2B-Instruct'  #@param {type:"string"}
OUT_DIR     = './svg_out'                   #@param {type:"string"}
MAX_TOKENS  = 3000                          #@param {type:"slider", min:1024, max:6000, step:256}

print(f'Prompts  : {len(PROMPTS)}')
print(f'Base     : {BASE_MODEL}')
print(f'Max tok  : {MAX_TOKENS}')
print(f'Out dir  : {OUT_DIR}')

In [ ]:
#@title 🛠️ Helper functions
import re, gc
from pathlib import Path
from IPython.display import display, SVG, HTML

SYSTEM_PROMPT = (
    'You are an expert SVG code generator. '
    'Given a text description, output ONLY valid SVG code. '
    'Rules:\n'
    '- Start with: <svg viewBox="0 0 200 200" xmlns="http://www.w3.org/2000/svg">\n'
    '- Use simple shapes: <rect>, <circle>, <ellipse>, <polygon>, <path>\n'
    '- Use solid hex fill colours (e.g., fill="#FF0000")\n'
    '- Keep it minimal: 5-20 elements\n- End with: </svg>\n'
    'Output the SVG code directly, no explanation.'
)

def repair_svg(svg: str) -> str:
    svg = svg.strip()
    m = re.search(r'<svg[\s>]', svg)
    if m:
        svg = svg[m.start():]
    open_g  = len(re.findall(r'<g\b[^>]*>', svg))
    close_g = len(re.findall(r'</g>', svg))
    svg += '</g>' * max(0, open_g - close_g)
    if not svg.rstrip().endswith('</svg>'):
        svg = svg.rstrip()
        svg += '\n</svg>' if svg.endswith('>') or svg.endswith('/>') else '" fill="#000000"/>\n</svg>'
    return svg

def svg_quality(svg: str) -> dict:
    paths     = len(re.findall(r'<path\b',     svg))
    rects     = len(re.findall(r'<rect\b',     svg))
    circles   = len(re.findall(r'<circle\b',   svg))
    ellipses  = len(re.findall(r'<ellipse\b',  svg))
    polygons  = len(re.findall(r'<polygon\b',  svg))
    polylines = len(re.findall(r'<polyline\b', svg))
    lines     = len(re.findall(r'<line\b',     svg))
    elements  = paths + rects + circles + ellipses + polygons + polylines + lines
    d_attrs   = re.findall(r'd="([^"]*)"', svg)
    max_d_len = max((len(d) for d in d_attrs), default=0)
    return {
        'elements': elements, 'paths': paths, 'rects': rects,
        'circles': circles, 'length': len(svg),
        'has_zeros':  bool(re.search(r'0{20,}', svg)),
        'has_repeat': bool(re.search(r'(\b\w+\b)(\s+\1){9,}', svg)),
        'is_potrace': max_d_len > 500,
    }

def show_svg(svg: str, label: str = ''):
    if label:
        display(HTML(f'<b>{label}</b>'))
    try:
        display(SVG(svg))
    except Exception:
        display(HTML(svg))

print('✅ Helpers ready')

In [ ]:
#@title 🤖 Load model + LoRA adapter
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print(f'Loading base model: {BASE_MODEL} (4-bit)...')
quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
)
base = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    quantization_config=quant,
    device_map='auto',
    trust_remote_code=True,
)
print(f'Loading LoRA adapter: {ADAPTER_PATH}...')
model = PeftModel.from_pretrained(base, ADAPTER_PATH)
model.eval()
tok = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
print(f'✅ Model ready  |  VRAM used: {torch.cuda.max_memory_allocated()/1e9:.1f} GB')

In [ ]:
#@title 🚀 Run inference on all prompts
import torch
from pathlib import Path

out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

stop_id = tok.encode('</svg>', add_special_tokens=False)
stop_id = stop_id[-1] if stop_id else tok.eos_token_id

results = []

for i, prompt in enumerate(PROMPTS):
    print(f'\n{"─"*55}')
    print(f'[{i+1}/{len(PROMPTS)}] {prompt}')

    chat = tok.apply_chat_template(
        [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': f'Generate SVG for: {prompt}'},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    ids = tok(chat, return_tensors='pt').to(model.device)

    with torch.inference_mode():
        out = model.generate(
            **ids,
            max_new_tokens=MAX_TOKENS,
            do_sample=False,
            repetition_penalty=1.05,  # gentle; 1.5 kills Potrace's repetitive coord data
            eos_token_id=stop_id,
        )

    raw = tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()

    # Strip markdown fences if present
    if '```' in raw:
        for part in raw.split('```'):
            p = part.strip().lstrip('svg').strip()
            if p.startswith('<svg'):
                raw = p
                break

    # Clip to <svg ... </svg>
    m = re.search(r'<svg[\s>]', raw)
    if m:
        raw = raw[m.start():]

    svg = repair_svg(raw)
    q   = svg_quality(svg)

    flags = []
    if q['has_zeros']:  flags.append('ZERO-LOOP')
    if q['has_repeat']: flags.append('WORD-REPEAT')
    if q['elements'] == 0: flags.append('NO-ELEMENTS')
    if q['is_potrace']:    flags.append('potrace-style')
    flag_str = '  ⚠ ' + ' | '.join(flags) if flags else '  ✓'

    print(f'  Length   : {q["length"]:,} chars')
    print(f'  Elements : {q["elements"]}  (paths={q["paths"]} rects={q["rects"]} circles={q["circles"]}){flag_str}')
    print(f'  Tail     : {raw[-60:]!r}')

    # Save SVG
    slug = re.sub(r'\W+', '_', prompt)[:40]
    svg_path = out_dir / f'{i:03d}_{slug}.svg'
    svg_path.write_text(svg, encoding='utf-8')
    print(f'  Saved    : {svg_path}')

    # Display inline
    show_svg(svg, prompt)

    results.append({'prompt': prompt, 'svg': svg, 'quality': q})

gc.collect()
torch.cuda.empty_cache()
print(f'\n{"═"*55}')
print(f'Done. {len(PROMPTS)} prompts → {OUT_DIR}')
print(f'{"═"*55}')

In [ ]:
#@title 📊 Summary
ok       = sum(1 for r in results if not r['quality']['has_zeros']
               and not r['quality']['has_repeat'] and r['quality']['elements'] > 0)
degen    = sum(1 for r in results if r['quality']['has_zeros'] or r['quality']['has_repeat'])
empty    = sum(1 for r in results if r['quality']['elements'] == 0)
potrace  = sum(1 for r in results if r['quality']['is_potrace'])
avg_len  = sum(r['quality']['length'] for r in results) / len(results)

print(f'Valid          : {ok}/{len(results)}')
print(f'Degenerate     : {degen}')
print(f'Empty          : {empty}')
print(f'Potrace-style  : {potrace}')
print(f'Avg SVG length : {avg_len:.0f} chars')

In [ ]:
#@title 💾 Download all SVGs as zip
import subprocess
from google.colab import files

zip_path = 'svg_out.zip'
subprocess.run(['zip', '-qr', zip_path, OUT_DIR], check=True)
files.download(zip_path)
print(f'✅ Downloading {zip_path}')